# Exploratory frozen-JEPA probe on StrokePiG

This notebook evaluates the three EMA `target_encoder` checkpoints in `outputs/repaired-jepa-seed7-v2` without retraining any encoder. Each C3D walking trial is mapped to the AMASS Core11 body-frame contract, resampled from 100 Hz to 30 Hz, divided into full 64-frame / 32-stride windows, and pooled into one 256-dimensional representation per participant.

The downstream target is the participant's signed propulsive-impulse asymmetry, `log(J_right / J_left)`, computed only from raw force plates. It uses event-matched, spatially verified, single-foot contacts. With two contacts per side this is **exploratory**, not confirmatory: this archive does not meet the project's four-contact, reliability, and n≥30 force gate. The encoder never receives force signals or force-derived cycles.

Controls use the same nested participant-level ridge protocol: pooled raw Core11 coordinates and a checkpoint-matched frozen random encoder. Results therefore test frozen representation transfer, not a stroke classifier or clinical deployment.

In [ ]:
from pathlib import Path
import os
import sys

import ezc3d
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src/gavd6_sjepa/gait_parity_jepa.py').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook inside the gavd6 repository')

PROJECT_DIR = find_project_root()
if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

from src.gavd6_sjepa.amass_core11_jepa import MIRROR_PAIRS, TIME_PATCH_FRAMES, WINDOW_FRAMES
from src.gavd6_sjepa.gavd_core11_probe import (
    AdapterConfig, _body_frame, _leg_length, _resample, build_probe_windows,
    frozen_target_encoder, parity_sequence_features, random_target_encoder,
    raw_coordinate_features,
)

DATA_DIR = PROJECT_DIR / 'data/50_StrokePiG'
CHECKPOINT_DIR = PROJECT_DIR / 'outputs/repaired-jepa-seed7-v2'
OUTPUT_DIR = PROJECT_DIR / 'work/artifacts/strokepig_frozen_jepa_probe'
DEVICE = os.getenv('GAIT_PARITY_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')

assert DATA_DIR.is_dir(), DATA_DIR
assert CHECKPOINT_DIR.is_dir(), CHECKPOINT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'project={PROJECT_DIR}')
print(f'device={DEVICE}; output={OUTPUT_DIR}')

## C3D → Core11 and raw-force contract

The Vicon Plug-in Gait point labels are segment-frame landmarks, not all joint centres. The correct Core11 mapping is `PELO`, `LFEP/RFEP`, `LFEO/RFEO`, `LTIO/RTIO`, and the measured `LHEE/RHEE/LTOE/RTOE`, for pelvis, hips, knees, ankles, heels, and forefeet respectively. In particular, `LFEO` is near the knee and `LFOO` is near the toe; neither is used as a hip or ankle.

For the independent target, raw `Force.Fx/Fy/Fz` plate channels are baseline-corrected and sign-inverted to express ground-on-body force. The positive component along the movement-derived forward axis is integrated over each strict single-foot stance. C3D foot-strike events match force episodes but do not determine encoder windows.

In [ ]:
CORE11_POINTS = ('PELO', 'LFEP', 'RFEP', 'LFEO', 'RFEO', 'LTIO', 'RTIO',
                 'LHEE', 'RHEE', 'LTOE', 'RTOE')
FORCE_THRESHOLD_N = 20.0
MIN_CONTACT_SECONDS = 0.10
MAX_EVENT_OFFSET_SECONDS = 0.15
MIN_CONTACTS_PER_SIDE = 2  # exploratory only; confirmatory analysis requires >=4
ALPHAS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3, 1e4)

def clean_label(value):
    return str(value).split(':')[-1].replace('Force.', '').strip()

def foot_strikes(c3d):
    event = c3d['parameters'].get('EVENT', {})
    labels = event.get('LABELS', {}).get('value', [])
    contexts = event.get('CONTEXTS', {}).get('value', [])
    times = np.asarray(event.get('TIMES', {}).get('value', np.empty((2, 0))), float)
    if times.ndim != 2 or times.shape[0] < 2:
        return []
    return [(float(time), str(context)[:1].upper())
            for label, context, time in zip(labels, contexts, times[1])
            if str(label).lower() == 'foot strike' and str(context)[:1].upper() in {'L', 'R'}]

def boolean_runs(mask, minimum_samples):
    padded = np.r_[False, np.asarray(mask, dtype=bool), False]
    edges = np.flatnonzero(padded[1:] != padded[:-1])
    return [(int(left), int(right)) for left, right in zip(edges[::2], edges[1::2])
            if right - left >= minimum_samples]

def point_in_convex_polygon(point, polygon):
    cross_products = []
    for first, second in zip(polygon, np.roll(polygon, -1, axis=0)):
        edge, relative = second - first, point - first
        cross_products.append(edge[0] * relative[1] - edge[1] * relative[0])
    cross_products = np.asarray(cross_products)
    return bool(np.all(cross_products >= -1e-6) or np.all(cross_products <= 1e-6))

def feet_on_plate(points, point_index, corners, frame, plate):
    polygon = corners[:2, :, plate].T
    centre = polygon.mean(axis=0)
    output = {}
    for side in ('L', 'R'):
        heel = points[:2, point_index[f'{side}HEE'], frame]
        toe = points[:2, point_index[f'{side}TOE'], frame]
        midpoint = (heel + toe) / 2
        finite = np.isfinite(np.r_[heel, toe]).all()
        output[side] = {
            'inside': bool(finite and (point_in_convex_polygon(heel, polygon)
                                       or point_in_convex_polygon(toe, polygon)
                                       or point_in_convex_polygon(midpoint, polygon))),
            'distance': float(np.linalg.norm(midpoint - centre)) if finite else np.inf,
        }
    return output

def extract_trial(path):
    c3d = ezc3d.c3d(str(path))
    events = foot_strikes(c3d)
    if 'cal' in path.stem.lower() or not events:
        return None, []

    point_labels = [clean_label(label) for label in c3d['parameters']['POINT']['LABELS']['value']]
    point_index = {label: index for index, label in enumerate(point_labels)}
    required = set(CORE11_POINTS) | {'LHEE', 'RHEE', 'LTOE', 'RTOE'}
    missing = required.difference(point_index)
    if missing:
        raise ValueError(f'{path}: missing points {sorted(missing)}')

    points = np.asarray(c3d['data']['points'][:3], dtype=np.float32)
    core11 = points[:, [point_index[label] for label in CORE11_POINTS]].transpose(2, 1, 0)
    valid = np.isfinite(core11).all(axis=2)
    if np.any(valid.mean(axis=0) < 0.95):
        raise ValueError(f'{path}: insufficient Core11 validity')

    leg_length = _leg_length(core11, valid)
    transform, frame_method = _body_frame(core11, valid, leg_length, AdapterConfig())
    body = ((core11 - core11[:, :1]) / np.float32(leg_length)) @ transform.T
    body[~valid] = 0.0
    point_rate = float(c3d['header']['points']['frame_rate'])
    canonical, canonical_valid = _resample(body, valid, point_rate, 30.0)

    record = {
        'participant': path.parent.name,
        'sequence_id': path.relative_to(DATA_DIR).with_suffix('').as_posix(),
        'video_id': path.parent.name,
        'condition': 'stroke',
        'coordinates': canonical,
        'valid': canonical_valid,
        'frame_method': frame_method,
        'leg_length_mm': leg_length,
    }

    analog_labels = [clean_label(label) for label in c3d['parameters']['ANALOG']['LABELS']['value']]
    analog_index = {label: index for index, label in enumerate(analog_labels)}
    analog = np.asarray(c3d['data']['analogs'][0], dtype=float)
    analog_rate = float(c3d['header']['analogs']['frame_rate'])
    analog_start = float(c3d['header']['analogs']['first_frame']) / analog_rate
    point_start = float(c3d['header']['points']['first_frame']) / point_rate
    corners = np.asarray(c3d['parameters']['FORCE_PLATFORM']['CORNERS']['value'], dtype=float)
    candidates = []
    for plate in range(corners.shape[2]):
        force_channels = [analog_index[f'F{axis}{plate + 1}'] for axis in 'xyz']
        raw_force = analog[force_channels]
        ground_force = -(raw_force - np.median(raw_force, axis=1, keepdims=True))
        for start, stop in boolean_runs(ground_force[2] > FORCE_THRESHOLD_N,
                                        int(round(MIN_CONTACT_SECONDS * analog_rate))):
            if start == 0 or stop == ground_force.shape[1]:
                continue
            peak = start + int(np.argmax(ground_force[2, start:stop]))
            onset = analog_start + start / analog_rate
            event_time, side = min(events, key=lambda event: abs(event[0] - onset))
            if abs(event_time - onset) > MAX_EVENT_OFFSET_SECONDS:
                continue
            frame = int(round((analog_start + peak / analog_rate - point_start) * point_rate))
            frame = int(np.clip(frame, 0, points.shape[2] - 1))
            feet = feet_on_plate(points, point_index, corners, frame, plate)
            other_side = 'R' if side == 'L' else 'L'
            closest_side = min(('L', 'R'), key=lambda foot: feet[foot]['distance'])
            if not (feet[side]['inside'] and not feet[other_side]['inside'] and closest_side == side):
                continue
            fore_aft = ground_force[:2, start:stop].T @ transform[0, :2]
            impulse = float(np.trapezoid(np.clip(fore_aft, 0.0, None), dx=1.0 / analog_rate))
            if impulse > 0.0:
                candidates.append({
                    'participant': path.parent.name, 'trial': record['sequence_id'],
                    'side': side, 'event_time': event_time, 'plate': plate + 1,
                    'peak_vertical_n': float(ground_force[2, peak]), 'impulse_ns': impulse,
                })

    # A strike may excite two plates; retain its strongest vertical-force episode.
    deduplicated = {}
    for candidate in candidates:
        key = (candidate['trial'], candidate['event_time'], candidate['side'])
        if key not in deduplicated or candidate['peak_vertical_n'] > deduplicated[key]['peak_vertical_n']:
            deduplicated[key] = candidate
    return record, list(deduplicated.values())

In [ ]:
records, contact_rows = [], []
for path in sorted(DATA_DIR.glob('TVC*/*.c3d')):
    record, contacts = extract_trial(path)
    if record is not None:
        records.append(record)
        contact_rows.extend(contacts)

# Transformer windows must be complete: exclude the one short trial rather than zero-pad it.
records = [record for record in records if len(record['coordinates']) >= WINDOW_FRAMES]
retained_trials = {record['sequence_id'] for record in records}
contacts = pd.DataFrame(contact_rows)
contacts = contacts.loc[contacts.trial.isin(retained_trials)].copy()

side_summary = contacts.groupby(['participant', 'side']).agg(
    contacts=('impulse_ns', 'size'), impulse_ns=('impulse_ns', 'mean')
).unstack('side')
side_summary.columns = ['_'.join(column) for column in side_summary.columns]
eligible = side_summary.loc[(side_summary.get('contacts_L', 0) >= MIN_CONTACTS_PER_SIDE)
                            & (side_summary.get('contacts_R', 0) >= MIN_CONTACTS_PER_SIDE)].copy()
eligible['target_log_right_over_left'] = np.log(
    eligible['impulse_ns_R'] / eligible['impulse_ns_L']
)
target_table = eligible.reset_index().sort_values('participant').reset_index(drop=True)
participants = target_table.participant.tolist()
records = [record for record in records if record['participant'] in set(participants)]

assert len(participants) >= 10, 'Too few exploratory bilateral-force participants'
assert np.isfinite(target_table.target_log_right_over_left).all()
print(f'walking trials retained: {len(records)}; strict contacts: {len(contacts)}')
print(f'exploratory participants with >= {MIN_CONTACTS_PER_SIDE} contacts/side: {len(participants)}')
display(target_table)
display(pd.DataFrame(records).groupby('frame_method').agg(
    trials=('sequence_id', 'size'), median_leg_length_mm=('leg_length_mm', 'median')
))

## Frozen representations

Each participant's feature vector pools all of that participant's valid Core11 tokens across eligible walking trials. The paired-orbit representation is `[even mean, even std, odd mean, odd std]`, giving `4 × 64 = 256` features. EMA encoders are set to evaluation mode and every parameter has `requires_grad=False`.

In [ ]:
windows, token_valid, trial_indices, window_table = build_probe_windows(records, AdapterConfig())
participant_index = {participant: index for index, participant in enumerate(participants)}
window_participants = np.asarray(
    [participant_index[records[index]['participant']] for index in trial_indices], dtype=np.int64
)
assert windows.shape[1:] == (64, 11, 3)
assert token_valid.shape[1] == (WINDOW_FRAMES // TIME_PATCH_FRAMES) * 11
print(f'paired-valid Core11 token fraction: {token_valid.mean():.4f}')

feature_sets = {
    'raw_core11': raw_coordinate_features(
        windows, token_valid, window_participants, len(participants)
    )
}
checkpoints = sorted(CHECKPOINT_DIR.glob('*_best.pt'))
assert len(checkpoints) == 3, [path.name for path in checkpoints]
encoder_audit = []
for index, checkpoint in enumerate(checkpoints):
    encoder, metadata = frozen_target_encoder(checkpoint)
    trained_name = f"ema_{metadata['variant']}"
    feature_sets[trained_name] = parity_sequence_features(
        encoder, windows, token_valid, window_participants, len(participants),
        device=DEVICE, batch_size=32,
    )
    random_encoder = random_target_encoder(metadata, seed=1000 + index)
    feature_sets[f"random_{metadata['variant']}"] = parity_sequence_features(
        random_encoder, windows, token_valid, window_participants, len(participants),
        device=DEVICE, batch_size=32,
    )
    encoder_audit.append({
        'checkpoint': checkpoint.name, 'variant': metadata['variant'],
        'embed_dim': encoder.config.embed_dim,
        'frozen_trainable_parameters': sum(p.numel() for p in encoder.parameters() if p.requires_grad),
        'features_finite': bool(np.isfinite(feature_sets[trained_name]).all()),
    })

assert all(np.isfinite(features).all() for features in feature_sets.values())
assert all(features.shape[0] == len(participants) for features in feature_sets.values())
display(pd.DataFrame(encoder_audit))
print(f'windows={len(windows)}; participants={len(participants)}')

## Identical nested participant-level ridge probes

One row is one participant, so every outer-test participant and every inner-validation participant is unseen by its fitted scaler and ridge model. The inner selection criterion is MAE. Out-of-fold MAE is primary; out-of-fold $R^2$, correlation, and calibration slope are descriptive because the exploratory cohort is small.

In [ ]:
def nested_ridge(feature_sets, targets, participant_ids, seed=42, outer_splits=5, inner_splits=4):
    targets = np.asarray(targets, dtype=float)
    participant_ids = np.asarray(participant_ids)
    outer = KFold(n_splits=min(outer_splits, len(targets)), shuffle=True, random_state=seed)
    summary_rows, fold_rows, prediction_rows = [], [], []
    for representation, features in feature_sets.items():
        features = np.asarray(features, dtype=float)
        if features.shape[0] != len(targets) or not np.isfinite(features).all():
            raise ValueError(f'Invalid features for {representation}: {features.shape}')
        oof = np.full(len(targets), np.nan)
        for fold, (train, test) in enumerate(outer.split(features)):
            inner = KFold(n_splits=min(inner_splits, len(train)), shuffle=True, random_state=seed + fold + 1)
            pipeline = Pipeline([('scale', StandardScaler()), ('ridge', Ridge())])
            search = GridSearchCV(
                pipeline, {'ridge__alpha': ALPHAS}, cv=inner,
                scoring='neg_mean_absolute_error', refit=True,
            ).fit(features[train], targets[train])
            prediction = search.predict(features[test])
            oof[test] = prediction
            fold_rows.append({
                'representation': representation, 'fold': fold,
                'test_participants': ','.join(participant_ids[test]),
                'alpha': float(search.best_params_['ridge__alpha']),
                'mae': mean_absolute_error(targets[test], prediction),
            })
        pearson = pearsonr(targets, oof).statistic if np.std(oof) > 0 else np.nan
        spearman = spearmanr(targets, oof).statistic if np.std(oof) > 0 else np.nan
        slope = np.polyfit(oof, targets, 1)[0] if np.std(oof) > 0 else np.nan
        summary_rows.append({
            'representation': representation, 'participants': len(targets),
            'feature_dimension': features.shape[1],
            'mae': mean_absolute_error(targets, oof),
            'rmse': mean_squared_error(targets, oof) ** 0.5,
            'r2': r2_score(targets, oof),
            'pearson_r': pearson, 'spearman_rho': spearman,
            'calibration_slope': slope,
        })
        prediction_rows.extend({
            'representation': representation, 'participant': participant,
            'target_log_right_over_left': target, 'prediction': prediction,
        } for participant, target, prediction in zip(participant_ids, targets, oof))
    return (pd.DataFrame(summary_rows), pd.DataFrame(fold_rows), pd.DataFrame(prediction_rows))

summary, folds, predictions = nested_ridge(
    feature_sets, target_table.target_log_right_over_left.to_numpy(), participants
)
summary = summary.sort_values('mae').reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / 'nested_probe_summary.csv', index=False)
folds.to_csv(OUTPUT_DIR / 'nested_probe_folds.csv', index=False)
predictions.to_csv(OUTPUT_DIR / 'nested_probe_predictions.csv', index=False)
target_table.to_csv(OUTPUT_DIR / 'participant_targets.csv', index=False)
contacts.to_csv(OUTPUT_DIR / 'strict_force_contacts.csv', index=False)

display(summary)
display(folds.pivot(index='fold', columns='representation', values='mae'))
print('Exploratory only: two contacts/side and the current archive do not satisfy the confirmatory force-quality gate.')
print(f'Wrote results to {OUTPUT_DIR}')